This script is used to calculate TCW stats.

In [1]:
# import functions
# OS interaction and time
import os
import sys
import cftime
import datetime
import time
import glob
import dask
import dask.bag as db
import calendar
import importlib

# math and data
import math
import numpy as np
import netCDF4 as nc
import xarray as xr
import scipy as sp
import scipy.linalg
from scipy.signal import detrend
import pandas as pd
import pickle as pickle
from sklearn import linear_model
import matplotlib.patches as mpatches
from shapely.geometry.polygon import LinearRing
import statsmodels.stats.multitest as multitest

# random
from IPython.display import display
from IPython.display import HTML
import IPython.core.display as di # Example: di.display_html('<h3>%s:</h3>' % str, raw=True)

# paths to various directories
my_era5_path = '/glade/u/home/zcleveland/scratch/ERA5/'  # path to subset data
misc_data_path = '/glade/u/home/zcleveland/scratch/misc_data/'  # path to misc data
plot_out_path = '/glade/u/home/zcleveland/NAM_soil-moisture/ERA5_analysis/plots/'  # path to generated plots
scripts_main_path = '/glade/u/home/zcleveland/NAM_soil-moisture/scripts_main/'  # path to my dicts, lists, and functions

# import variable lists and dictionaries
if scripts_main_path not in sys.path:
    sys.path.insert(0, scripts_main_path)  # path to file containing these lists/dicts
if 'get_var_data' in sys.modules:
    importlib.reload(sys.modules['get_var_data'])
if 'my_functions' in sys.modules:
    importlib.reload(sys.modules['my_functions'])
if 'my_dictionaries' in sys.modules:
    importlib.reload(sys.modules['my_dictionaries'])

# import common functions that I've created
from get_var_data import get_var_data, get_var_files, open_var_data, subset_var_data, time_to_year_month_avg, time_to_year_month_sum, time_to_year_month
from my_functions import month_num_to_name, ensure_var_list

# import lists and dictionaries
from my_dictionaries import (
sfc_instan_list, sfc_accumu_list, pl_var_list, derived_var_list, invar_var_list,
NAM_var_list, region_avg_list, flux_var_list, vector_var_list, misc_var_list,
var_dict, var_units, region_avg_dict, region_avg_coords, region_colors_dict
)

In [6]:
# calculate 25th and 75th quantile, median, and mean for the max and min annual
# total column water values over the dsw

# initialize data lists
max_list = []
min_list = []

# open datasets
files = glob.glob(f'{my_era5_path}dsw/*/tcw_*_dsw.nc')
files.sort()
tcw = xr.open_mfdataset(files)

# calculate yearly min/max
min_min = tcw['TCW_AVG'].groupby('time.year').min()
max_max = tcw['TCW_AVG'].groupby('time.year').max()

# rechunk the data array along the 'year' dimension into a single chunk
min_min = min_min.chunk({'year': -1})
max_max = max_max.chunk({'year': -1})

# create datasets for tcw_min and tcw_max
tcw_min = xr.Dataset()
tcw_max = xr.Dataset()

# calculate statistics on min/max lists and store them into new datasets

# min
tcw_min['Q25'] = min_min.quantile(0.25, dim='year') # 25th percentile
tcw_min['Q75'] = min_min.quantile(0.75, dim='year') # 75th percentile
tcw_min['MEDIAN'] = min_min.quantile(0.5, dim='year') # median
tcw_min['MEAN'] = min_min.mean(dim='year') # mean

# max
tcw_max['Q25'] = max_max.quantile(0.25, dim='year') # 25th percentile
tcw_max['Q75'] = max_max.quantile(0.75, dim='year') # 75th percentile
tcw_max['MEDIAN'] = max_max.quantile(0.5, dim='year') # median
tcw_max['MEAN'] = max_max.mean(dim='year') # mean

# save datasets to netcdf files
print(tcw_min)
print(tcw_max)
tcw_min.to_netcdf(f'{my_era5_path}dsw/tcw_min_stats.nc')
tcw_max.to_netcdf(f'{my_era5_path}dsw/tcw_max_stats.nc')

<xarray.Dataset>
Dimensions:    (latitude: 81, longitude: 81)
Coordinates:
  * latitude   (latitude) float64 40.0 39.75 39.5 39.25 ... 20.5 20.25 20.0
  * longitude  (longitude) float64 240.0 240.2 240.5 240.8 ... 259.5 259.8 260.0
    quantile   float64 0.25
Data variables:
    Q25        (latitude, longitude) float64 dask.array<chunksize=(81, 81), meta=np.ndarray>
    Q75        (latitude, longitude) float64 dask.array<chunksize=(81, 81), meta=np.ndarray>
    MEDIAN     (latitude, longitude) float64 dask.array<chunksize=(81, 81), meta=np.ndarray>
    MEAN       (latitude, longitude) float32 dask.array<chunksize=(81, 81), meta=np.ndarray>
<xarray.Dataset>
Dimensions:    (latitude: 81, longitude: 81)
Coordinates:
  * latitude   (latitude) float64 40.0 39.75 39.5 39.25 ... 20.5 20.25 20.0
  * longitude  (longitude) float64 240.0 240.2 240.5 240.8 ... 259.5 259.8 260.0
    quantile   float64 0.25
Data variables:
    Q25        (latitude, longitude) float64 dask.array<chunksize=(81, 81), 